# 04a - PCA Foundations

Datasets often have many features, making them difficult to reason about, visualize, or understand. Which features matter? Which provide similar information? Can we describe the observations more simply while preserving useful differences among them?

An observation can have many measurements without having equally many distinct things to tell us. Several measurements may rise and fall together. A smaller set of coordinates can sometimes describe most of the differences among observations.

**Dimensionality reduction** reduces the number of features used to represent each observation. There are two broad approaches:

- **Feature selection** keeps a subset of the original features. The retained features keep their original meanings.
- **Feature extraction** constructs new features from the original ones. For dimensionality reduction, we keep fewer new features than we started with.

Today we will look at **principal component analysis**, or **PCA**, a useful form of feature extraction. PCA combines all of the original features into new representations based on dimensions that preserve as much variation as possible. The number of dimensions, or components, we keep determines the number of resulting features.

We will first see PCA in a nutshell, then explain how covariance, eigenvectors, and eigenvalues identify its directions. A four-observation example will walk through the calculation. Housing data will give us a richer example of what the resulting components mean.

## 1.0 PCA in a Nutshell

The cloud below contains 200 observations with two measurements each. It stretches along a diagonal: the measurements tend to increase together.

This example uses the same generated data as [VanderPlas's introduction to PCA](https://jakevdp.github.io/PythonDataScienceHandbook/05.09-principal-component-analysis.html).

In [ ]:
import hashlib
import io
import tarfile
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

plt.rcParams.update(
    {"figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False}
)
BLUE, ORANGE, PURPLE = "#245c91", "#b65313", "#75529a"

In [ ]:
nutshell_rng = np.random.RandomState(1)
nutshell_points = np.dot(nutshell_rng.rand(2, 2), nutshell_rng.randn(2, 200)).T

fig, axis = plt.subplots(figsize=(5, 3.9), layout="constrained")
axis.scatter(*nutshell_points.T, color=BLUE, s=15, alpha=0.45)
axis.set(xlabel="Measurement 1", ylabel="Measurement 2", xlim=(-3, 3), ylim=(-1.5, 1.5))
axis.set_aspect("equal")
axis.set_title("Two measurements per observation")
plt.show()

### 1.1 Pause: Best Representation?

What is the best representation of this data in a single dimension?

##### Answer

One option is to flatten the cloud onto the horizontal axis, keeping only measurement 1. Another is to flatten it onto the vertical axis, keeping only measurement 2.

What does each choice preserve, and what would it lose?

### 1.2 Quantifying the Question

In order to answer these questions, we can think in terms of total variance in the data. Geometrically, variance in one dimension is the sum of the squared distances from each observation to the mean along that dimension, divided by $(n-1)$. Total variance is the sum of those variances across all dimensions.

$$
\text{Total variance}
= s_{M_1}^{2} + s_{M_2}^{2}
= \frac{1}{n-1}\sum_{i=1}^{n}
\left[(M_{1i}-\bar M_1)^2 + (M_{2i}-\bar M_2)^2\right]
$$

Here, $n$ is the number of observations, and $\bar M_1$ and $\bar M_2$ are the means of the two measurements. Total variance is the sum of their individual sample variances.

In [ ]:
nutshell_centered = nutshell_points - nutshell_points.mean(axis=0)
nutshell_total_variance = (nutshell_centered**2).sum() / (len(nutshell_points) - 1)
print(f"Total variance of the original measurements: {nutshell_total_variance:.5f}")

### 1.3 Choosing Components

In the graph below, the orange points along the bottom show each observation's horizontal coordinate; the purple points along the left show its vertical coordinate. Each row of points is a separate possible one-dimensional representation. They are placed at the edges so we can compare them with the original cloud.

In [ ]:
fig, axis = plt.subplots(figsize=(5, 3.9), layout="constrained")
axis.scatter(*nutshell_points.T, color=BLUE, s=15, alpha=0.35)
axis.scatter(
    nutshell_points[:, 0], np.full(len(nutshell_points), -1.35),
    color=ORANGE, s=12, alpha=0.55,
)
axis.scatter(
    np.full(len(nutshell_points), -2.85), nutshell_points[:, 1],
    color=PURPLE, s=12, alpha=0.55,
)
axis.set(xlabel="Measurement 1", ylabel="Measurement 2", xlim=(-3, 3), ylim=(-1.5, 1.5))
axis.set_aspect("equal")
axis.set_title("Keep only the horizontal or vertical coordinate")
plt.show()

Let's compare the variance "captured" by each of these reductions:

In [ ]:
nutshell_measurement_variances = nutshell_points.var(axis=0, ddof=1)
nutshell_axis_comparison = pd.DataFrame(
    {
        "Variance retained": nutshell_measurement_variances,
        "Original variance retained (%)": (
            100 * nutshell_measurement_variances / nutshell_total_variance
        ),
    },
    index=["Measurement 1 only", "Measurement 2 only"],
)
nutshell_axis_comparison.round(
    {"Variance retained": 5, "Original variance retained (%)": 2}
)

Keeping measurement 1 retains 0.68218, or 87.35 percent of the original total variance. Keeping measurement 2 retains 0.09883, or 12.65 percent. Each representation uses one number per observation, but the horizontal coordinate preserves more of the original spread. These comparisons use the original measurement scales shown in the plots.

Neither original axis follows the cloud's long direction. Could a different coordinate preserve more of its spread? That is the fundamental concept of PCA.

### 1.4 A Better Way?

A simpler representation is useful only if it still lets us distinguish observations in meaningful ways. Finding similar observations, discovering groups, and identifying unusual cases all depend on those differences. If we flatten the data in a way that erases them, we may also erase the patterns we want to discover.

PCA finds the direction with the greatest spread, then additional orthogonal directions describing the remaining spread. These directions, one per component, give us new axes for locating each observation.

The left panel shows those directions. The right panel shows what happens when we keep only the coordinate along the longer direction: the cloud flattens onto a line. Positions along that line still distinguish observations, but differences across it disappear. The faded points show the original cloud for comparison.

In [ ]:
nutshell_model = PCA(n_components=2).fit(nutshell_points)
nutshell_scores = nutshell_model.transform(nutshell_points)
nutshell_flattened = (
    nutshell_scores[:, :1] @ nutshell_model.components_[:1] + nutshell_model.mean_
)

fig, axes = plt.subplots(1, 2, figsize=(12.5 * 2 / 3, 3.9), layout="constrained")
for axis in axes:
    axis.set(xlabel="Measurement 1", ylabel="Measurement 2", xlim=(-3, 3), ylim=(-1.5, 1.5))
    axis.set_aspect("equal")
axes[0].scatter(*nutshell_points.T, color=BLUE, s=15, alpha=0.45)
for component, variance, color, label in zip(
    nutshell_model.components_, nutshell_model.explained_variance_,
    [ORANGE, PURPLE], ["Greatest spread", "Remaining spread"], strict=True,
):
    direction = component * np.sign(component[0])
    endpoint = nutshell_model.mean_ + 2 * np.sqrt(variance) * direction
    axes[0].annotate(
        "", xy=endpoint, xytext=nutshell_model.mean_,
        arrowprops=dict(arrowstyle="->", color=color, lw=2, mutation_scale=14),
    )
    axes[0].plot([], [], color=color, label=label)
axes[0].legend(fontsize=8, loc="upper left")
axes[0].set_title("Find the main directions")
axes[1].scatter(*nutshell_points.T, color=BLUE, s=15, alpha=0.15)
axes[1].scatter(*nutshell_flattened.T, color=ORANGE, s=15, alpha=0.65)
axes[1].set_title("Keep one direction")
plt.show()

PCA finds the new axes. Keeping both coordinates would preserve every point's position; keeping only one reduces the description to one number per observation. Here, that preserves the cloud's long spread while giving up its width.

In [ ]:
nutshell_pc1_variance = nutshell_scores[:, 0].var(ddof=1)
nutshell_comparison = nutshell_axis_comparison.copy()
nutshell_comparison.loc["First principal component only"] = [
    nutshell_pc1_variance,
    100 * nutshell_pc1_variance / nutshell_total_variance,
]
nutshell_comparison.round(
    {"Variance retained": 5, "Original variance retained (%)": 2}
)

The first principal component retains 0.76253, or 97.63 percent of the original total variance. That compares with 87.35 percent for measurement 1 alone and 12.65 percent for measurement 2 alone. All three representations use one number per observation, but PCA's coordinate preserves the most variance. The remaining 2.37 percent is the variation across the cloud's narrow width that disappears when we keep only the first component.

Next we will examine how PCA finds these directions, then work through the calculation with four observations.

## 2.0 Finding the Principal Directions

We need a new set of directions for describing our observations. The first should capture the greatest possible variance. After finding it, PCA searches among all directions perpendicular to it and chooses the one capturing the most remaining variance.

With $p$ original features, a complete set has $p$ mutually perpendicular directions, one for each component. We then choose how many components to keep.

### 2.1 Why Perpendicular Directions?

Why perpendicular? A direction only slightly tilted from the first would largely measure the same variation again. We want the next direction to describe variation the first has not captured.

In three dimensions, the directions perpendicular to the first axis form a plane. PCA searches within that plane for the direction with the greatest remaining variance. Once it chooses that second axis, only one axis perpendicular to both remains. Each subsequent direction must be perpendicular to every direction already chosen.

The diagram places the first principal direction vertically so its perpendicular plane is easy to see. The second direction is chosen within that plane; the third must be perpendicular to both.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, PURPLE = "#245c91", "#b65313", "#75529a"
fig = plt.figure(figsize=(6, 4.5), layout="constrained")
axis = fig.add_subplot(111, projection="3d")
plane_x, plane_y = np.meshgrid([-1.2, 1.2], [-1.2, 1.2])
axis.plot_surface(plane_x, plane_y, np.zeros_like(plane_x), color=BLUE, alpha=0.15)
for vector, color, label in [
    ([0, 0, 1.5], BLUE, "First direction"),
    ([1.2, 0.6, 0], ORANGE, "Second direction"),
    ([-0.6, 1.2, 0], PURPLE, "Third direction"),
]:
    axis.quiver(0, 0, 0, *vector, color=color, linewidth=2.5, arrow_length_ratio=0.12)
    axis.text(*(np.array(vector) * 1.12), label, color=color, fontsize=9)
axis.set(xlim=(-1.7, 1.7), ylim=(-1.7, 1.7), zlim=(-0.4, 1.9))
axis.set_box_aspect((1, 1, 0.8))
axis.view_init(elev=30, azim=-25)
axis.set_axis_off()
axis.set_title("Search the Plane Perpendicular to the First Direction")
plt.show()

To find the next component, restrict the search to directions perpendicular to every component already chosen. In three dimensions, choosing the first two leaves only one possible axis for the third. In higher dimensions, several directions may still be available; PCA chooses the one capturing the greatest remaining variance.

### 2.2 The Covariance Matrix

To find these directions, we need to know how much each original feature varies and how the features vary together. If two features tend to increase together, a direction combining increases in both may capture more spread than either original axis.

The *covariance matrix* collects this information. Each diagonal entry is a feature's variance. Each off-diagonal entry is the covariance between two features, describing whether they tend to move together or in opposite directions.

Each covariance compares two features across the same observations. The covariance matrix collects these pairwise comparisons for all features, including each feature's covariance with itself, which is its variance.

For two features, $x_1$ and $x_2$, the covariance matrix is

$$
S=\begin{bmatrix}
\operatorname{Var}(x_1) & \operatorname{Cov}(x_1,x_2)\\
\operatorname{Cov}(x_1,x_2) & \operatorname{Var}(x_2)
\end{bmatrix}
$$

For $n$ observations, sample variance and covariance are calculated as follows:

$$
\operatorname{Var}(x_1)=\frac{1}{n-1}\sum_{i=1}^{n}(x_{1i}-\bar x_1)^2
$$

$$
\operatorname{Cov}(x_1,x_2)=\frac{1}{n-1}\sum_{i=1}^{n}(x_{1i}-\bar x_1)(x_{2i}-\bar x_2)
$$

The bars denote feature means, and $i$ identifies an observation. Variance sums squared deviations from one feature's mean. Covariance sums products of the two features' deviations. Calculate $\operatorname{Var}(x_2)$ in the same way as $\operatorname{Var}(x_1)$, using feature 2.

The covariance appears twice because the covariance of $x_1$ with $x_2$ is the same as the covariance of $x_2$ with $x_1$. This makes the matrix symmetric: exchanging its rows and columns leaves it unchanged.

This matrix lets us calculate how much variance would be captured along any proposed direction. First, we need to describe that direction and the coordinate it produces.

### 2.3 Variance Along a Proposed Direction

We need a way to describe the variance of the data for any given direction. For two features, a direction vector has two coefficients, $a$ and $b$. These tell us how much of each feature to combine. Write them as a column:

$$
w=\begin{bmatrix}a\\b\end{bmatrix}
$$

To calculate the *coordinate* for any observation along the direction $w$, multiply its centered feature values by its coefficients and add:

$$
z=ax_1+bx_2
$$

Each observation has a coordinate $z$ along the direction $w$. Across all observations, these coordinates form a new feature. Its variance measures how spread out the observations are along that direction.

Above, $x_1$ and $x_2$ denote the centered features. Centering does not change their variances or covariance, so the same matrix $S$ applies.

To compare directions fairly, make each direction vector one unit long. Otherwise, doubling both coefficients would double every new coordinate and quadruple its variance, even though the direction had not changed. For our two coefficients, unit length means

$$
a^2+b^2=1
$$

We can calculate the variance of $z$ in two operations, without first calculating $z$ for every observation.

First, multiply the covariance matrix by the direction vector. Matrix multiplication combines each row of $S$ using the coefficients $a$ and $b$:

$$
Sw=\begin{bmatrix}
a\operatorname{Var}(x_1)+b\operatorname{Cov}(x_1,x_2)\\
a\operatorname{Cov}(x_1,x_2)+b\operatorname{Var}(x_2)
\end{bmatrix}
$$

The result is another vector. Its entries are the covariances between each original feature and the new feature $z$:

$$
Sw=\begin{bmatrix}\operatorname{Cov}(x_1,z)\\\operatorname{Cov}(x_2,z)\end{bmatrix}
$$

The [Course Addenda and Errata](../../pages/course-addenda-and-errata.md#04a-covariance-with-a-combined-feature) shows why these two expressions are equivalent.

Second, take the dot product with the same direction vector. Multiply the first entry by $a$, the second by $b$, and add. The notation $w^\mathsf{T}$ turns the column vector into a row so we can write this operation as

$$
w^\mathsf{T}(Sw)
=a\operatorname{Cov}(x_1,z)+b\operatorname{Cov}(x_2,z)
=\operatorname{Cov}(ax_1+bx_2,z)
$$

But $ax_1+bx_2$ is $z$. We have therefore calculated the covariance of $z$ with itself, which is its variance. Substituting gives the final form:

$$
\operatorname{Var}(z)=w^\mathsf{T}Sw
$$

The variance along direction $w$ is calculated by multiplying the covariance matrix by $w$, then multiplying the result by $w^\mathsf{T}$. The result is one number: the variance of the observations' coordinates $z$ along that unit direction.

The same approach works for any number of features. Now we have a way to calculate the variance in any direction. We still need to find the best directions without trying every possibility.

### 2.4 Eigenvectors and Eigenvalues

That is where *eigenvectors* and *eigenvalues* enter. A matrix can change both a vector's length and its direction. An eigenvector identifies a special direction: applying the matrix keeps that nonzero vector on the same line. Its eigenvalue tells us the scaling factor. They come as a pair: a direction preserved by the matrix and its scaling factor.

Below, the same covariance matrix acts on two vectors. The first changes direction. The eigenvector stays on its original line. Dashed arrows show the input vectors; solid arrows show the results.

In [ ]:
# A schematic covariance matrix chosen only to illustrate matrix action.
illustration_covariance = np.array([[2.0, 1.0], [1.0, 2.0]])
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8), layout="constrained")
for axis, vector, title in zip(
    axes,
    [np.array([1.0, 0.0]), np.array([1.0, 1.0]) / np.sqrt(2)],
    ["A Vector That Changes Direction", "An Eigenvector Stays on Its Line"],
    strict=True,
):
    result = illustration_covariance @ vector
    axis.axhline(0, color="0.85", linewidth=1)
    axis.axvline(0, color="0.85", linewidth=1)
    for endpoint, color, style, label, width in [
        (result, ORANGE, "-", "After Matrix Multiplication", 2.5),
        (vector, BLUE, "--", "Original Vector", 2),
    ]:
        axis.annotate(
            "", xy=endpoint, xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color=color, linestyle=style, lw=width),
        )
        axis.plot([], [], color=color, linestyle=style, label=label)
    axis.set(xlim=(-0.3, 2.7), ylim=(-0.3, 2.7), xticks=[], yticks=[])
    axis.set_aspect("equal")
    axis.set_title(title, fontsize=11)
    axis.legend(loc="upper left", fontsize=8)
    axis.spines[["top", "right", "bottom", "left"]].set_visible(False)
plt.show()

It is worth noting that Eigenvectors and eigenvalues are useful *properties* that can be calculated for any square matrix. They are not specific to PCA or covariance matricies.

For the covariance matrix $S$, an eigenvector $w$ and its eigenvalue $\lambda$ satisfy

$$
Sw=\lambda w
$$

The left side applies the matrix to the vector. The right side multiplies every entry of that vector by the same number, $\lambda$. Both operations give the same result. The vector supplies the direction; the eigenvalue supplies the scaling factor.

We already know how to calculate variance along a unit direction: multiply by $S$, then take the dot product with that direction. For an eigenvector, the first operation gives $\lambda w$. Substituting that result into the second operation gives

$$
\operatorname{Var}(z)=w^\mathsf{T}Sw=w^\mathsf{T}(\lambda w)
=\lambda(w^\mathsf{T}w)
$$

The dot product $w^\mathsf{T}w$ is the sum of the squared coefficients. Because the vector has unit length, that sum is one. Therefore,

$$
\operatorname{Var}(z)=\lambda
$$

The variance along the unit eigenvector $w$ equals its corresponding eigenvalue of the covariance matrix. Equivalently, the variance of the scores $z$ is that eigenvalue.

For a unit eigenvector of the covariance matrix, the eigenvalue is both the matrix's scaling factor and the variance captured along that direction. Different eigenvectors can have different eigenvalues, giving us a way to compare their captured variance.

### 2.5 Choosing the Leading Directions

We start with only the covariance matrix. Solving its eigenvalue equation identifies directions that the matrix preserves and the scaling factor for each. There can be several solutions, so one fixed matrix can yield several eigenvector–eigenvalue pairs.

With $p$ features, the covariance matrix allows a complete set of $p$ mutually perpendicular unit eigenvectors. Each has a corresponding eigenvalue, although some eigenvalues may be equal or zero. Our variance equation applies separately to each pair.

We therefore calculate the pairs and rank them by eigenvalue. The largest identifies the first principal direction; the next-largest identifies the second, and so on. The first captures the greatest possible variance; each subsequent direction captures the greatest remaining variance perpendicular to the earlier directions.

Each observation can then be described by its coordinates along those axes. Keeping only the leading coordinates gives us a representation with fewer dimensions that retains as much variance as possible.

### 2.6 Calculating Eigenvalues and Eigenvectors

We know the covariance matrix $S$. We want to find the eigenvalues and eigenvectors that satisfy

$$
Sw=\lambda w
$$

A standard result from linear algebra lets us find the eigenvalues by solving

$$
\det(S-\lambda I)=0
$$

Here, $I$ is the identity matrix, so subtracting $\lambda I$ subtracts $\lambda$ from each diagonal entry of $S$. A determinant is a single number calculated from a square matrix. For a two-by-two matrix, multiply the diagonal entries and subtract the product of the off-diagonal entries:

$$
\det\begin{bmatrix}a&b\\c&d\end{bmatrix}=ad-bc
$$

Setting the determinant to zero gives us an equation to solve for the eigenvalues, $\lambda$; we will use this result without deriving it.

For a two-feature covariance matrix,

$$
S=\begin{bmatrix}v_1&c\\c&v_2\end{bmatrix}
$$

where $v_1$ and $v_2$ are the feature variances and $c$ is their covariance. The equation becomes

$$
(v_1-\lambda)(v_2-\lambda)-c^2=0
$$

This is a quadratic equation giving the two eigenvalues.

Then, find the directions. Substitute each eigenvalue back into $Sw=\lambda w$ and solve for the relationship between the vector's coefficients. Divide the resulting vector by its length to make it a unit eigenvector.

We will work through this with actual numbers next. For larger matrices, Python handles these calculations.

## 3.0 A Worked Example

We will find the principal directions for four observations with two features, then use those directions to describe each observation with one coordinate. We will treat the original feature scales as comparable.

In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA

example = pd.DataFrame(
    {"Feature 1": [13.0, 15.0, 5.0, 7.0], "Feature 2": [24.0, 20.0, 20.0, 16.0]},
    index=pd.Index(["A", "B", "C", "D"], name="Observation"),
)
example

Our first direction should capture the greatest spread in these observations. With two features, only one perpendicular axis will remain after we choose the first.

### 3.1 Centering the Measurements

Subtract each feature's mean from its values. The means are 10 and 20, so observation A moves from $(13,24)$ to $(3,4)$.

In [ ]:
feature_means = example.mean()
centered = example - feature_means
display(feature_means.rename("Mean").to_frame())
centered

The centered observations are A: $(3,4)$, B: $(5,0)$, C: $(-5,0)$, and D: $(-3,-4)$. Both plots use identical axis limits and tick marks. The faint points in the right panel show the original positions: subtracting the means shifts the cloud to the origin without changing its shape or the distances between points.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.3, 3.9), layout="constrained")
for axis, data, title in zip(
    axes, [example, centered], ["Original Measurements", "Centered Measurements"],
    strict=True,
):
    if axis is axes[1]:
        axis.scatter(*example.to_numpy().T, color=BLUE, alpha=0.15)
    axis.scatter(*data.to_numpy().T, color=BLUE)
    for name, point in data.iterrows():
        axis.annotate(name, point.to_numpy() + [0.18, 0.18])
    axis.axhline(0, color="0.8", linewidth=1)
    axis.axvline(0, color="0.8", linewidth=1)
    axis.set(xlim=(-7, 17), ylim=(-6, 26),
             xticks=[-5, 0, 5, 10, 15], yticks=[-5, 0, 5, 10, 15, 20, 25])
    axis.set_aspect("equal")
    axis.set_title(title)
axes[0].set(xlabel="Feature 1", ylabel="Feature 2")
axes[1].set(xlabel="Centered Feature 1", ylabel="Centered Feature 2")
plt.show()

### 3.2 Building the Covariance Matrix

Following section 2.2, calculate the variance of each feature and their covariance. With four observations, the sample calculations divide by three:

$$
\operatorname{Var}(x_1)=\frac{3^2+5^2+(-5)^2+(-3)^2}{3}=\frac{68}{3}
$$

$$
\operatorname{Var}(x_2)=\frac{4^2+0^2+0^2+(-4)^2}{3}=\frac{32}{3}
$$

Feature 1 has greater variance: its squared deviations sum to 68, compared with 32 for feature 2.

$$
\operatorname{Cov}(x_1,x_2)=\frac{(3)(4)+(5)(0)+(-5)(0)+(-3)(-4)}{3}=8
$$

Put the variances on the diagonal and the covariance in the two off-diagonal positions:

$$
S=\begin{bmatrix}68/3&8\\8&32/3\end{bmatrix}
$$

Covariance is symmetric: the covariance of feature 1 with feature 2 equals the covariance of feature 2 with feature 1. The off-diagonal entries therefore mirror each other.

The code repeats these calculations using the centered columns:

In [ ]:
x1 = centered["Feature 1"].to_numpy()
x2 = centered["Feature 2"].to_numpy()
denominator = len(example) - 1
v1 = (x1**2).sum() / denominator
v2 = (x2**2).sum() / denominator
c = (x1 * x2).sum() / denominator
covariance = np.array([[v1, c], [c, v2]])
display(pd.DataFrame(covariance, index=example.columns, columns=example.columns).round(4))
total_variance = v1 + v2
print(f"Original total variance: {total_variance:.5f}")

The original total variance is 100/3, approximately 33.33333.

### 3.3 Evaluating a Proposed Direction

One simple way to reduce two features to one is to keep feature 1 and discard feature 2. The unit direction vector $(1,0)$ does exactly that. We can calculate how much variance this choice retains before asking whether PCA can do better.

First, multiply the covariance matrix by that direction:

$$
Sw=\begin{bmatrix}68/3&8\\8&32/3\end{bmatrix}
\begin{bmatrix}1\\0\end{bmatrix}
=\begin{bmatrix}68/3\\8\end{bmatrix}
$$

Then multiply by the row vector containing the same coefficients:

$$
w^\mathsf{T}Sw=\begin{bmatrix}1&0\end{bmatrix}
\begin{bmatrix}68/3\\8\end{bmatrix}=\frac{68}{3}
$$

In [ ]:
candidate_direction = np.array([1.0, 0.0])
matrix_times_direction = covariance @ candidate_direction
candidate_variance = candidate_direction @ matrix_times_direction
print("Matrix times direction:", matrix_times_direction)
print(f"Variance along feature 1: {candidate_variance:.5f}")
print(f"Original variance retained: {100 * candidate_variance / total_variance:.2f}%")

Keeping feature 1 retains 68 percent of the total variance. Now we will calculate the eigenvalues and eigenvectors to find the best direction.

### 3.4 Calculating the Eigenvalues

Following section 2.6, subtract the unknown eigenvalue from each diagonal entry and set the determinant to zero:

$$
\left(\frac{68}{3}-\lambda\right)\left(\frac{32}{3}-\lambda\right)-8^2=0
$$

Expanding the products gives

$$
\lambda^2-\frac{100}{3}\lambda+\frac{1600}{9}=0
$$

Multiply by nine to remove the fractions, then factor:

$$
9\lambda^2-300\lambda+1600=(3\lambda-80)(3\lambda-20)=0
$$

Either factor can equal zero. Ordered from largest to smallest, the two eigenvalues are

$$
\lambda_1=\frac{80}{3}\qquad\lambda_2=\frac{20}{3}
$$

These are the variances along the two principal directions. We still need to find the directions themselves. Python can verify the roots using the variances and covariance calculated above:

In [ ]:
# Expanding the determinant gives: lambda**2 - (v1 + v2)*lambda + v1*v2 - c**2.
eigenvalues = np.sort(np.roots([1, -(v1 + v2), v1 * v2 - c**2]))[::-1]
pd.Series(eigenvalues, index=["PC1", "PC2"], name="Eigenvalue")

### 3.5 Calculating the Eigenvectors

For the largest eigenvalue, substitute 80/3 into $Sw=\lambda w$. Write the unknown direction as $(a,b)$. The first row gives

$$
\frac{68}{3}a+8b=\frac{80}{3}a
\quad\Longrightarrow\quad
8b=4a
\quad\Longrightarrow\quad b=\frac{a}{2}
$$

The second row gives the same relationship. We can choose any nonzero value for $a$, but choosing 1 makes the math easy. We will normalize the resulting vector afterward, so this choice does not affect the axis it defines.

With $a=1$, we get $(1,1/2)$. Its length is $\sqrt{1^2+(1/2)^2}=\sqrt{5}/2$. Dividing by that length gives

$$
w_1=\frac{1}{\sqrt{5}/2}\begin{bmatrix}1\\1/2\end{bmatrix}
=\frac{1}{\sqrt{5}}\begin{bmatrix}2\\1\end{bmatrix}
$$

For the other eigenvalue, substitute 20/3:

$$
\frac{68}{3}a+8b=\frac{20}{3}a
\quad\Longrightarrow\quad
8b=-16a
\quad\Longrightarrow\quad b=-2a
$$

Again choose $a=1$. The vector $(1,-2)$ has length $\sqrt{5}$, giving

$$
w_2=\frac{1}{\sqrt{5}}\begin{bmatrix}1\\-2\end{bmatrix}
$$

PC1 gives feature 1 twice the coefficient of feature 2. PC2 combines the features with opposite signs. These coefficients come from our calculated covariance matrix, rather than an assumption about the directions.

To repeat this calculation in Python, rearrange the equation $v_1a+cb=\lambda a$ to obtain $b=(\lambda-v_1)a/c$. Set $a=1$, calculate $b$ for each eigenvalue, and normalize each vector:

In [ ]:
directions = np.column_stack(
    [np.array([1.0, (value - v1) / c]) for value in eigenvalues]
)
directions = directions / np.linalg.norm(directions, axis=0)
pd.DataFrame(directions, index=example.columns, columns=["PC1", "PC2"]).round(4)

The arrows show these unit eigenvectors on the original measurement axes. Both start at the data mean, where the new axes pass through the cloud.

In [ ]:
fig, axis = plt.subplots(figsize=(4.8, 3.9), layout="constrained")
axis.scatter(*example.to_numpy().T, color=BLUE)
for name, point in example.iterrows():
    axis.annotate(name, point.to_numpy() + [0.18, 0.18])
origin = feature_means.to_numpy()
for j, color in enumerate([ORANGE, PURPLE]):
    direction = directions[:, j]
    line = origin[:, None] + direction[:, None] * np.array([-6, 6])
    axis.plot(*line, color=color, alpha=0.35, linewidth=1)
    axis.annotate("", xy=origin + direction, xytext=origin,
                  arrowprops=dict(arrowstyle="->", color=color, lw=2))
    axis.text(*(origin + 2 * direction), f"PC{j + 1}", color=color, fontsize=10)
axis.scatter(*origin, color="0.3", s=20)
axis.set(xlabel="Feature 1", ylabel="Feature 2", xlim=(4, 16), ylim=(14, 26))
axis.set_aspect("equal")
axis.set_title("Principal Directions Through the Mean")
plt.show()

The two columns contain the direction coefficients, often called *loadings*. We can check both the eigenvector relationship and perpendicularity:

In [ ]:
print("S times each direction equals its eigenvalue times that direction:",
      np.allclose(covariance @ directions, directions * eigenvalues))
print(f"Dot product of the two directions: {directions[:, 0] @ directions[:, 1]:.5f}")

Their dot product is zero, confirming that the directions are perpendicular.

### 3.6 Calculating Scores and Choosing a Component

The eigenvectors supply the coefficients; each centered observation supplies the feature values. As in section 2.3, multiply the values by the coefficients and add to obtain a coordinate, or *score*, along that direction.

Let $x_{A1}$ and $x_{A2}$ denote A's centered feature values, and let $w_{jk}$ denote feature $j$'s coefficient in component $k$. For observation A, whose centered measurements are $(3,4)$, the first eigenvector supplies coefficients $2/\sqrt{5}$ and $1/\sqrt{5}$:

$$
z_{A1}=x_{A1}w_{11}+x_{A2}w_{21}
=3\left(\frac{2}{\sqrt{5}}\right)+4\left(\frac{1}{\sqrt{5}}\right)
=\frac{10}{\sqrt{5}}\approx4.472
$$

The second eigenvector supplies coefficients $1/\sqrt{5}$ and $-2/\sqrt{5}$:

$$
z_{A2}=x_{A1}w_{12}+x_{A2}w_{22}
=3\left(\frac{1}{\sqrt{5}}\right)+4\left(\frac{-2}{\sqrt{5}}\right)
=\frac{-5}{\sqrt{5}}\approx-2.236
$$

The eigenvectors define the axes; these two scores locate A along those axes. Matrix multiplication applies the same calculation to every observation:

In [ ]:
scores = centered.to_numpy() @ directions
pd.DataFrame(scores, index=example.index, columns=["PC1 Score", "PC2 Score"]).round(4)

A and B share a positive PC1 score; C and D share a negative PC1 score. PC2 distinguishes the members of each pair. Although A and B have different original measurements, their weighted combinations along PC1 are equal.

Calculate the sample variance of each score column and compare it with the eigenvalues:

In [ ]:
score_variances = scores.var(axis=0, ddof=1)
pd.DataFrame(
    {"Eigenvalue": eigenvalues, "Score Variance": score_variances,
     "Original Variance Retained (%)": 100 * score_variances / total_variance},
    index=["PC1", "PC2"],
).round(4)

The variances are 80/3 and 20/3, matching the eigenvalues. Together they account for all the original variance. If we keep one component, we choose PC1: it retains 80 percent, compared with 68 percent for feature 1 alone. PC2 accounts for the remaining 20 percent.

### 3.7 The Same Calculation with Scikit-Learn

Scikit-learn fits PCA directly from the original measurements. It centers the data for us. Here we request one component and calculate the scores with `transform`:

In [ ]:
pca = PCA(n_components=1, svd_solver="full").fit(example)
library_scores = pca.transform(example)
display(pd.DataFrame(pca.components_.T, index=example.columns, columns=["PC1 Coefficient"]))
display(pd.DataFrame(library_scores, index=example.index, columns=["PC1 Score"]))
print(f"PC1 variance: {pca.explained_variance_[0]:.5f}")
print(f"Original variance retained: {100 * pca.explained_variance_ratio_[0]:.2f}%")

Scikit-learn reproduces the coefficients, scores, and retained variance from our hand calculation.

We used the covariance eigenproblem to understand the calculation. With `svd_solver="full"`, scikit-learn uses a different numerical route, singular value decomposition, to obtain the same principal component solution. We do not need that additional derivation to use or interpret these results.

In the real-data example, we will interpret richer components and reconstruct approximate original measurements to examine what a reduced representation preserves.

## 4.0 Interpreting and Evaluating a Representation

The four-point example made the calculation visible. With real measurements, we also need to explain what the components describe and whether the retained variation serves the analysis.

### 4.1 California Housing Measurements

The California housing data describe 20,640 census block groups from the 1990 census. Each row is a block group, not an individual home. We use eight features: median income, median housing age, total rooms, total bedrooms, population, households, latitude, and longitude.

Rooms, bedrooms, population, and households are totals within a block group. This distinction matters when interpreting a component that combines them. Median house value is excluded from this PCA. We are describing variation in the selected features without fitting to an outcome.

The data cell reads the accompanying CSV when it is available. Otherwise, it downloads the public source archive used by scikit-learn. The [dataset description](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) provides the source background; our feature set retains the original totals rather than converting them to averages per household.

In [ ]:
source_columns = [
    "longitude",
    "latitude",
    "median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "median_house_value",
]
housing_features = [
    "median_income",
    "median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "latitude",
    "longitude",
]
housing_labels = [
    "Median income",
    "Median housing age",
    "Total rooms",
    "Total bedrooms",
    "Population",
    "Households",
    "Latitude",
    "Longitude",
]
housing_candidates = [
    Path("data/04a-california-housing.csv"),
    Path("content/lectures/04a-pca-foundations/data/04a-california-housing.csv"),
]
housing_file = next((path for path in housing_candidates if path.is_file()), None)

if housing_file is not None:
    housing = pd.read_csv(housing_file)[housing_features]
else:
    housing_url = "https://ndownloader.figshare.com/files/5976036"
    with urlopen(housing_url, timeout=30) as response:
        housing_archive = response.read()
    expected_sha256 = "aaa5c9a6afe2225cc2aed2723682ae403280c4a3695a2ddda4ffb5d8215ea681"
    if hashlib.sha256(housing_archive).hexdigest() != expected_sha256:
        raise ValueError("The housing source archive does not match the expected data.")
    with tarfile.open(fileobj=io.BytesIO(housing_archive), mode="r:gz") as archive:
        housing = pd.read_csv(
            archive.extractfile("CaliforniaHousing/cal_housing.data"), names=source_columns
        )[housing_features]

housing.columns = housing_labels
if housing.shape != (20640, 8) or not np.isfinite(housing.to_numpy()).all():
    raise ValueError("Expected 20,640 complete observations on eight housing features.")
print(f"{len(housing):,} block groups; {housing.shape[1]} features")
housing.head().round(3)

### 4.2 Putting Features on Comparable Scales

In our earlier work with distance-based measures, we saw that feature scales must be consistent or comparable for distances to reflect the differences we intend to measure. Otherwise, a feature with large numerical values can dominate simply because of its units. The same concern applies here: PCA measures spread using squared deviations, so changing a feature's scale changes how much it contributes to total variance.

The housing features mix counts, years, income, and geographical coordinates. For this analysis, we give each feature equal initial variance by subtracting its mean and dividing by its sample standard deviation:

In [ ]:
housing_means = housing.mean()
housing_scales = housing.std(ddof=1)
housing_standardized = (housing - housing_means) / housing_scales
pd.DataFrame({
    "Original Standard Deviation": housing_scales,
    "Standardized Mean": housing_standardized.mean(),
    "Standardized Variance": housing_standardized.var(ddof=1),
}).round(3)

Every standardized feature has mean zero and variance one. PCA will therefore describe variation relative to each feature's usual spread. This is a choice about which differences should count. PCA centers its input automatically, but it does not standardize feature scales for us.

### 4.3 Creating and Fitting a PCA Model

`PCA(...)` creates an estimator object: a configurable model that has not yet learned anything from our data. The `n_components` argument controls how many components to keep. We will initially keep all eight so we can inspect the complete result before choosing a smaller number.

In [ ]:
housing_pca = PCA(n_components=housing_standardized.shape[1], svd_solver="full")
housing_pca

The constructor's `svd_solver="full"` setting selects the numerical method used for fitting. We use the same method as in the worked example.

Calling `.fit(data)` learns the feature means, principal directions, and variance captured by each component. It stores them on the estimator and returns that same fitted object. It does not return the transformed dataset.

In [ ]:
housing_pca.fit(housing_standardized)

Once fitted, the object has attributes containing what it learned:

| Attribute | What It Contains |
| --- | --- |
| `mean_` | The mean of each input feature, subtracted when transforming observations |
| `components_` | The coefficients defining each component, one component per row |
| `explained_variance_` | The variance captured by each component |
| `explained_variance_ratio_` | Each component's share of the total input variance |

The trailing underscore marks an attribute learned during fitting. Because we supplied standardized data, the fitted means are already approximately zero.

### 4.4 From Coefficients to Observation Scores

First inspect the coefficients. There are eight components and eight input features, so `components_` has eight rows and eight columns. The labels tell us which is which:

In [ ]:
housing_component_names = [f"PC{i + 1}" for i in range(housing_pca.n_components_)]
print("Coefficient array shape:", housing_pca.components_.shape)
pd.DataFrame(
    housing_pca.components_, index=housing_component_names, columns=housing.columns
).round(3)

Each row is a recipe for a new feature. For each observation, PCA multiplies its centered input values by that row's coefficients and adds the products. The resulting number is that observation's score on the component.

For example, each PC1 score combines the block group's standardized measurements using the coefficients learned during fitting. With coefficients rounded to three decimal places:

$$
\begin{aligned}
\text{PC1 score}\approx{}&0.045(\text{Median income})-0.218(\text{Median housing age})\\
&+0.484(\text{Total rooms})+0.491(\text{Total bedrooms})\\
&+0.472(\text{Population})+0.492(\text{Households})\\
&-0.073(\text{Latitude})+0.076(\text{Longitude})
\end{aligned}
$$

Every feature value in this equation is standardized, not in its original units. For one block group, multiply each standardized value by its coefficient and add the eight products to obtain its PC1 score. The other score columns use the same calculation with their own coefficient rows.

Use `.transform(data)` to calculate these scores. It applies the fitted means and directions rather than learning new ones:

In [ ]:
housing_model_scores = housing_pca.transform(housing_standardized)
print("Input shape:", housing_standardized.shape)
print("Score array shape:", housing_model_scores.shape)
pd.DataFrame(
    housing_model_scores, index=housing.index, columns=housing_component_names
).head().round(3)

Each score row is still one census block group. Its columns now describe positions along the principal directions instead of income, room totals, and the other original measurements. Keeping all eight components changes the coordinates but does not yet reduce the number of features.

For later observations, apply the same saved feature means and scales before calling the fitted model's `.transform()`. That keeps their coordinates comparable with the observations used to fit the model.

### 4.5 Components Defined by Feature Patterns

To interpret a component, look at which features have the largest coefficient magnitudes and whether their signs agree or oppose one another. Then use those feature meanings to describe what a high or low score represents.

For one block group, we can see how the inputs contribute to its PC1 score. Multiply each standardized value by its coefficient, then add the products:

In [ ]:
observation_standardized = housing_standardized.iloc[2913]
pc1_coefficients = pd.Series(housing_pca.components_[0], index=housing.columns)
observation_contributions = observation_standardized * pc1_coefficients
pd.DataFrame({
    "Standardized feature": observation_standardized,
    "PC1 coefficient": pc1_coefficients,
    "Contribution to score": observation_contributions,
}).round(3)

The contributions sum to a PC1 score of about 1.940. The block group's housing and population totals supply most of that positive score. The plots below show the coefficients shared by all observations, rather than this block group's individual contributions.

A component's entire set of coefficients can be multiplied by minus one without changing its axis. For the displays below, we orient each component so its largest-magnitude coefficient is positive, and orient its scores the same way. Relative signs within each component still matter.

In [ ]:
housing_directions = housing_pca.components_.T.copy()

# Give each displayed component's largest coefficient a positive sign.
for component in range(housing_pca.n_components_):
    largest = np.argmax(np.abs(housing_directions[:, component]))
    if housing_directions[largest, component] < 0:
        housing_directions[:, component] *= -1

housing_scores = housing_standardized.to_numpy() @ housing_directions
housing_coefficients = pd.DataFrame(
    housing_directions, index=housing.columns, columns=housing_component_names
)

fig, axes = plt.subplots(1, 5, figsize=(17, 3), sharey=True, layout="constrained")
for component, axis in enumerate(axes.flat):
    values = housing_directions[:, component]
    axis.barh(housing_labels, values, color=np.where(values >= 0, BLUE, ORANGE))
    axis.axvline(0, color="0.4", linewidth=1)
    axis.set(title=f"PC{component + 1} Coefficients", xlabel="Coefficient", xlim=(-1, 1))
    axis.grid(axis="x", alpha=0.2)
axes[0].invert_yaxis()
plt.show()

PC1 combines large positive coefficients on rooms, bedrooms, population, and households. A shared housing-and-population scale description fits: higher scores tend to identify block groups with larger totals. The negative housing-age coefficient adds a secondary association with newer housing. This does not describe individual home size or population density.

### 4.6 Pause: Describe the Remaining Four Components

Which features define PC2, PC3, PC4, and PC5 most strongly? Give each component a short description supported by the coefficient pattern, then explain what a high score would tend to indicate.

##### Answer

PC2 has strong positive latitude and negative longitude coefficients. It describes a geographical contrast, with higher scores tending toward northern and western locations in this orientation. The component is a combination of features, so no single measurement determines every observation's score.

PC3 is dominated by positive median income and negative median housing age coefficients. Higher scores tend to identify block groups with higher incomes and newer housing. Negative coefficients on bedrooms, population, and households add a smaller contribution from lower totals on those measures.

PC4 combines a strong positive median housing age coefficient with a positive median income coefficient. Higher scores tend to identify block groups with older housing and higher incomes. Unlike PC3, housing age and income contribute in the same direction here.

PC5 contrasts a strong positive population coefficient with negative room, bedroom, and household coefficients. Higher scores tend to identify block groups whose population is large relative to their housing totals. This is a weighted contrast of standardized measurements, not a calculated people-per-household ratio.

### 4.7 Choosing How Many Components to Keep

The fitted model gives us all eight directions in order of variance captured. Choosing `n_components` means deciding how far down that ordered list to go. Keeping more components preserves more variation; keeping fewer gives a simpler representation.

The individual percentages show what each component contributes. The cumulative percentage adds the contributions of the leading components:

In [ ]:
housing_variance_summary = pd.DataFrame({
    "Variance": housing_pca.explained_variance_,
    "Variance Retained (%)": 100 * housing_pca.explained_variance_ratio_,
    "Cumulative Retained (%)": 100 * np.cumsum(housing_pca.explained_variance_ratio_),
}, index=housing_component_names)
housing_variance_summary.round(2)

Each standardized input feature had variance one, so the original total variance is eight. The component variances also sum to eight. The first two components retain about 72.68 percent of that total.

The left plot is a *scree plot*: component variance against component number. A bend where the curve starts to flatten can suggest that additional components contribute relatively little. The right plot shows cumulative retention so we can inspect explicit percentage targets.

In [ ]:
component_numbers = np.arange(1, housing_pca.n_components_ + 1)
fig, axes = plt.subplots(1, 2, figsize=(8.3, 3.9), layout="constrained")
axes[0].plot(component_numbers, housing_pca.explained_variance_, "o-", color=BLUE)
axes[0].set(title="Variance by Component", xlabel="Component", ylabel="Variance", ylim=(0, None))
axes[1].plot(
    component_numbers, housing_variance_summary["Cumulative Retained (%)"], "o-", color=ORANGE
)
axes[1].axhline(90, color="0.5", linestyle="--", linewidth=1)
axes[1].set(
    title="Cumulative Variance Retained", xlabel="Number of Components",
    ylabel="Percent", ylim=(0, 105),
)
for axis in axes:
    axis.set_xticks(component_numbers)
    axis.grid(alpha=0.2)
plt.show()

For example, two components would allow a two-dimensional plot. Keeping enough for 90 percent variance is a different goal. We can calculate how many leading components reach that threshold:

In [ ]:
housing_target_fraction = 0.90
housing_selected_n = int(
    np.searchsorted(np.cumsum(housing_pca.explained_variance_ratio_), housing_target_fraction) + 1
)
print(f"Components needed for 90% variance: {housing_selected_n}")
display(housing_variance_summary.iloc[[1, housing_selected_n - 1, -1]].round(2))

There is no universally correct percentage. A threshold measures retained spread, while our analytical purpose determines whether the retained differences are useful. Component interpretation helps us assess that choice.

Once we choose a number, pass it as `n_components` to obtain a reduced score table directly:

In [ ]:
housing_reduced_pca = PCA(n_components=housing_selected_n, svd_solver="full")
housing_reduced_pca.fit(housing_standardized)
housing_reduced_scores = housing_reduced_pca.transform(housing_standardized)
print("Reduced score array shape:", housing_reduced_scores.shape)
pd.DataFrame(
    housing_reduced_scores, index=housing.index,
    columns=housing_component_names[:housing_selected_n],
).head().round(3)

The number of observations stays the same. The chosen number of components determines the number of new features. Retained variance summarizes the dataset as a whole; it does not tell us whether every feature important to our analysis is well represented.

### 4.8 Pause: Choose Evidence for the Next Decision

An analyst wants to compare block groups partly by income and proposes keeping only PC1 and PC2 because they retain about 73 percent of total variance. Looking back at the coefficient plots, what might this representation miss? Which additional component would you examine?

##### Answer

Income has small coefficients in PC1 and PC2, but a large coefficient in PC3. Keeping only the first two components could underrepresent income differences. Examine PC3 before deciding how many components to keep; the overall retained percentage alone does not establish that the representation suits this comparison.

### 4.9 Rules of Thumb for Choosing the Number of Components

There is no single rule that gives the best number of components for every analysis. These guidelines provide starting points:

- **Look for an elbow in the scree plot.** Keep the components through the point where the steep decline becomes a relatively flat tail. If the curve flattens after PC4, four components is a candidate. Some plots have no clear elbow.
- **Set a cumulative variance target.** Choose the fewest components that reach a target such as 80, 90, or 95 percent. The target expresses how much total variation you want to preserve; it does not guarantee that particular differences of interest survive.
- **Use the Kaiser rule for standardized features.** When every input feature has variance one, consider keeping components with eigenvalues greater than one. Each then captures more variance than one original standardized feature. This is a rough heuristic, not a decisive cutoff.
- **Check the analytical purpose.** Examine whether the retained components represent the differences you care about. Two or three components may suit a visualization; a comparison or clustering analysis may need more.

Use these rules together. If they suggest different counts, compare those candidate representations rather than treating any one rule as definitive. A scree plot measures diminishing variance contributions, not how easily the components can be named or interpreted.

## 5.0 From a Fitted Transformation to an Analytical Choice

PCA learns perpendicular directions from the covariance of the supplied features. Their eigenvalues describe the variance of the component scores. Retaining the leading directions preserves the most variance for that number of linear coordinates.

The coefficients explain how the coordinates are constructed, and the original feature meanings help us interpret them. Scaling determines which variation is emphasized. The next step is to choose how many components the analysis needs and examine whether the resulting representation is useful and stable.

## 6.0 References

- VanderPlas, *Python Data Science Handbook*, second edition, Chapter 45, especially pp. 463–467: the visual progression from original points to principal coordinates and reconstruction. The [online first-edition chapter](https://jakevdp.github.io/PythonDataScienceHandbook/05.09-principal-component-analysis.html) is an accessible companion. The opening cloud uses VanderPlas's generated data; the worked example uses a separate four-observation dataset.
- James et al., *An Introduction to Statistical Learning with Applications in Python*, §12.2.1–12.2.4 and §12.5.1: components, scores, approximation, retained variance, scaling, and the PCA lab.
- Zaki and Meira, *Data Mining and Machine Learning*, second edition, [Chapter 7](https://dataminingbook.info/book_html/chap7/book.html), especially §7.2.1: projected variance and the covariance eigenproblem.
- Jeffrey S. Smith, *Dimension-reduction Methods*, INSY 7120, Spring 2019, slides 15–21: component interpretation and housing profiles. This notebook recomputes its housing results from the full public dataset.
- [scikit-learn PCA documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html) and [California housing dataset documentation](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset): fitted outputs and data provenance.